In [84]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict
from langchain_groq import ChatGroq

In [85]:
class batsmanstate(TypedDict):
    runs : int 
    balls : int
    fours : int
    six : int
    sr : float
    bpb : float
    boundary_percent : float
    summary : str

In [86]:
def calculate_sr(state : batsmanstate) :
    sr=(state['runs']/state['balls'])*100

    return {'sr':sr}

In [87]:
def calculate_bpb(state:batsmanstate):
    bpb=state['balls']/(state['fours']+state['six'])

    return {'bpb':bpb}

In [88]:
def calculate_boundary_percent(state:batsmanstate):
    boundary_percent=(((4*state['fours'])+(6*state['six']))/state['runs'])*100

    return {'boundary_percent':boundary_percent}

In [89]:
def summary(state:batsmanstate)-> batsmanstate:
    summary=f"""
    strike rate = {state['sr']} \n
    balls_per_boundary={state['bpb']} \n
    boundary_percentage={state['boundary_percent']} 
    """

    return {'summary':summary}

In [90]:
graph=StateGraph(batsmanstate)

In [91]:
graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_boundary_percent',calculate_boundary_percent)
graph.add_node('summary',summary)

In [92]:
graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_boundary_percent')
graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_boundary_percent','summary')
graph.add_edge('summary',END)

In [93]:
workflow=graph.compile()

In [94]:
initial_state={'runs' : 100,
    'balls' : 40,
    'fours' : 5,
    'six' : 10}

final_state=workflow.invoke(initial_state)

print(final_state)

print(final_state['summary'])

{'runs': 100, 'balls': 40, 'fours': 5, 'six': 10, 'sr': 250.0, 'bpb': 2.6666666666666665, 'boundary_percent': 80.0, 'summary': '\n    strike rate = 250.0 \n\n    balls_per_boundary=2.6666666666666665 \n\n    boundary_percentage=80.0 \n    '}

    strike rate = 250.0 

    balls_per_boundary=2.6666666666666665 

    boundary_percentage=80.0 
    
